In [1]:
# Install dependencies (uncomment if running fresh)
%pip install nltk rouge-score sacrebleu bert-score pandas matplotlib -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import nltk
import pandas as pd
import matplotlib.pyplot as plt

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

pd.set_option("display.max_colwidth", 120)

In [3]:
synthetic_data = [
    {
        "id": 1,
        "case": "exact_match",
        "question": "What is the capital of France?",
        "reference": "The capital of France is Paris.",
        "candidate": "The capital of France is Paris.",
    },
    {
        "id": 2,
        "case": "good_paraphrase",
        "question": "What is the capital of France?",
        "reference": "The capital of France is Paris.",
        "candidate": "Paris is France's capital city.",
    },
    {
        "id": 3,
        "case": "partially_correct",
        "question": "Summarize the causes of World War I.",
        "reference": "World War I was caused by militarism, alliances, imperialism, and nationalism, triggered by the assassination of Archduke Franz Ferdinand.",
        "candidate": "World War I started because of the assassination of Archduke Franz Ferdinand and rising nationalism in Europe.",
    },
    {
        "id": 4,
        "case": "hallucinated",
        "question": "What is the capital of France?",
        "reference": "The capital of France is Paris.",
        "candidate": "The capital of France is Lyon, a city known for its cuisine.",
    },
    {
        "id": 5,
        "case": "off_topic",
        "question": "What is the boiling point of water at sea level?",
        "reference": "Water boils at 100 degrees Celsius at sea level.",
        "candidate": "Water freezes at 0 degrees Celsius.",
    },
    {
        "id": 6,
        "case": "good_summary",
        "question": "Summarize: The stock market fell sharply today after the central bank raised interest rates, citing persistent inflation concerns.",
        "reference": "Stocks dropped after the central bank hiked interest rates due to inflation worries.",
        "candidate": "Markets declined following an interest rate hike from the central bank amid inflation concerns.",
    },
]

df = pd.DataFrame(synthetic_data)
df

,id,case,question,reference,candidate
0,1,exact_match,What is the capital of France?,The capital of France is Paris.,The capital of France is Paris.
1,2,good_paraphrase,What is the capital of France?,The capital of France is Paris.,Paris is France's capital city.
2,3,partially_correct,Summarize the causes of World War I.,"World War I was caused by militarism, alliances, imperialism, and nationalism, triggered by the assassination of Arc...",World War I started because of the assassination of Archduke Franz Ferdinand and rising nationalism in Europe.
3,4,hallucinated,What is the capital of France?,The capital of France is Paris.,"The capital of France is Lyon, a city known for its cuisine."
4,5,off_topic,What is the boiling point of water at sea level?,Water boils at 100 degrees Celsius at sea level.,Water freezes at 0 degrees Celsius.
5,6,good_summary,"Summarize: The stock market fell sharply today after the central bank raised interest rates, citing persistent infla...",Stocks dropped after the central bank hiked interest rates due to inflation worries.,Markets declined following an interest rate hike from the central bank amid inflation concerns.


In [4]:
# BLEU Score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

smoothie = SmoothingFunction().method1

def compute_bleu(reference: str, candidate: str) -> float:
    ref_tokens = [reference.lower().split()]
    cand_tokens = candidate.lower().split()
    return sentence_bleu(ref_tokens, cand_tokens, smoothing_function=smoothie)

df["bleu"] = df.apply(lambda r: compute_bleu(r["reference"], r["candidate"]), axis=1)
df[["case", "reference", "candidate", "bleu"]]

,case,reference,candidate,bleu
0,exact_match,The capital of France is Paris.,The capital of France is Paris.,1.000000
1,good_paraphrase,The capital of France is Paris.,Paris is France's capital city.,0.052312
2,partially_correct,"World War I was caused by militarism, alliances, imperialism, and nationalism, triggered by the assassination of Arc...",World War I started because of the assassination of Archduke Franz Ferdinand and rising nationalism in Europe.,0.262168
3,hallucinated,The capital of France is Paris.,"The capital of France is Lyon, a city known for its cuisine.",0.317023
4,off_topic,Water boils at 100 degrees Celsius at sea level.,Water freezes at 0 degrees Celsius.,0.032588
5,good_summary,Stocks dropped after the central bank hiked interest rates due to inflation worries.,Markets declined following an interest rate hike from the central bank amid inflation concerns.,0.080323


In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def compute_rouge(reference: str, candidate: str) -> dict:
    scores = scorer.score(reference, candidate)
    return {
        "rouge1_f": scores["rouge1"].fmeasure,
        "rouge2_f": scores["rouge2"].fmeasure,
        "rougeL_f": scores["rougeL"].fmeasure,
    }

rouge_results = df.apply(lambda r: compute_rouge(r["reference"], r["candidate"]), axis=1)
df = pd.concat([df, pd.DataFrame(list(rouge_results))], axis=1)
df[["case", "rouge1_f", "rouge2_f", "rougeL_f"]]